# ExoPie: Quickstart Guide

Welcome to **ExoPie**, an open-source Python library designed for modeling and simulating the interior structures of exoplanets. 

This quickstart tutorial will walk you through the basics of the library, including:
1. Calculating the Rocky Threshold Radius (RTR).
2. Modeling the interior structure of a rocky exoplanet based on its mass and radius.
3. Applying specific compositional constraints to the core and mantle.
4. Saving your models and visualizing the parameter space.

Let's start by importing the necessary packages.

In [3]:
import numpy as np
import pickle
import os
from scipy.interpolate import RegularGridInterpolator

In [57]:
import numpy as np
from exopie.tools import chemistry
from scipy.optimize import minimize

class star:
    """
    Initializes and models the properties of a host star to estimate 
    the interior composition of its orbiting exoplanets.
    """

    def __init__(self,Fe=[0,0.04],Si=[0,0.04],Mg=[0,0.04],
                 interior_ratio_constraints=None,
                 abundances_normalization=None,
                 N=50000, **kwargs):
        """
        Initialize the HostStar object.

        Args:
            star_abundance (list): Log abundances of [Fe, Si, Mg, Ca, Al, Ni].
                                   Use None for missing elements.
            interior_ratio_constraints (list, optional): Elemental ratios to constrain.
            abundances_normalization (list, optional): Sollar normalization coeficients.
            N (int): Number of Monte Carlo samples to draw.
            **kwargs: Additional parameters for xSi, xFe, and xCore_trace.
        """
        if interior_ratio_constraints is None:
            interior_ratio_constraints = ['Fe/Si', 'Fe/Mg', 'Mg/Si', 'Ni/Fe', 'Ca/Mg', 'Al/Mg']
        if abundances_normalization is None:
            # Assume Asplund 2021 solar abundances
            abundances_normalization = [7.46, 7.55, 7.51, 6.30, 6.43, 6.20]

        self.N = N
        self.interior_ratio_constraints = interior_ratio_constraints
        elements = ['Fe', 'Si', 'Mg', 'Ca', 'Al', 'Ni']
        for i, item in enumerate(elements):
            star_abundance = kwargs.get(item, [-2,0])
            print(kwargs.get(item, [-2,0]))
            if len(star_abundance)<=3:
                star_abundance = self._set_parameter(mu=star_abundance[0], sigma=star_abundance[1:])
            setattr(self,item,star_abundance)

        current_abundances = [self.Fe,self.Si,self.Mg,self.Ca,self.Al,self.Ni]
        # Atomic masses for Fe, Si, Mg, Ca, Al, Ni (in kg/mol)
        mu = [55.85e-3, 28.09e-3, 24.31e-3, 40.08e-3, 26.98e-3, 58.69e-3] 
        star = {}
        for i, item in enumerate(elements):
            star[item] = 10**(current_abundances[i]+abundances_normalization[i])*mu[i]

        self.dr_star_ratios = {}
        for i,item in enumerate(self.interior_ratio_constraints):
            self.dr_star_ratios[item] = eval(item, star.copy())
        self.xSi = kwargs.get('xSi', [0,0.2])
        self.xFe = kwargs.get('xFe', [0,0.2])
        self.xCore_trace = kwargs.get('xCore_trace', 0.02)
        if len(self.xSi)<=3:
            self.xSi = self._set_parameter(mu=self.xSi[0], sigma=self.xSi[1:])
        if len(self.xFe)<=3:
            self.xFe = self._set_parameter(mu=self.xFe[0], sigma=self.xFe[1:])
        
    def star_to_planet(self,tol=1e-8):
        model_param = np.zeros([self.N,8])
        planet_data = np.zeros([self.N,6])
        for i in range(self.N):
            xfe,xsi = self.xFe[i],self.xSi[i]
            star_ratios = {item: self.dr_star_ratios[item] for item in self.dr_star_ratios.keys()}
            
            res = minimize(self._minerology_residual,[0.325,0.2,0,0,0],args=[star_ratios,xfe,xsi],tol=tol,
                        bounds=[[1e-15,1-1e-15],[1e-15,0.5],[0,0.2],[0,0.2],[0,0.2]])
            
            if res.success:
                cmf, Xmgsi, xNi, xAl, xCa = res.x
                model_param[i] = cmf,xsi,xfe,xNi,xAl,xCa,xWu,xSiO2
                femf, simf, mgmf, camf, almf, nimf = chemistry(cmf,xSi=xsi,xFe=xfe,trace_core=self.xCore_trace,
                                    xNi=xNi,xAl=xAl,xCa=xCa,xWu=0,xSiO2=0)
                xSiO2, xWu = (0, Xmgsi) if star_ratios['Mg/Si'] > mgmf / simf else (Xmgsi, 0)
                planet_data[i] = chemistry(cmf,xSi=xsi,xFe=xfe,trace_core=self.xCore_trace,
                                        xNi=xNi,xAl=xAl,xCa=xCa,xWu=xWu,xSiO2=xSiO2)
        
                
                
            else:
                model_param[i] = np.repeat(np.nan,8)
                planet_data[i] = np.repeat(np.nan,6)
        
        for i,item in enumerate(['FeMF','SiMF','MgMF','CaMF','AlMF','NiMF']):
            setattr(self,item,planet_data[:,i])
        for i,item in enumerate(['CMF','xSi','xFe','xNi','xAl','xCa','xWu','xSiO2']):
            setattr(self,item,model_param[:,i])

    def _minerology_residual(self,x,args):
        cmf, Xmgsi, xNi, xAl, xCa = x
        dr_star_ratios,xFe,xSi = args
        femf,simf,mgmf,camf,almf,nimf = chemistry(cmf,xSi=self.xSi,xFe=self.xFe,trace_core=self.xCore_trace,
                                xNi=xNi,xAl=xAl,xCa=xCa,xWu=0,xSiO2=0)
        xSiO2, xWu = (0, Xmgsi) if dr_star_ratios['Mg/Si'] > mgmf / simf else (Xmgsi, 0)
        femf,simf,mgmf,camf,almf,nimf = chemistry(cmf,xSi=xSi,xFe=xFe,trace_core=self.xCore_trace,
                                xNi=xNi,xAl=xAl,xCa=xCa,xWu=xWu,xSiO2=xSiO2)
        dr_planet = {'Fe': femf, 'Mg': mgmf, 'Si': simf, 'Ca': camf, 'Al': almf, 'Ni': nimf}
        res = 0
        for item in self.interior_ratio_constraints:
            res+=np.sum(dr_star_ratios[item]-eval(item, dr_planet.copy()))**2/1e-6
        return res 


    def _set_parameter(self, mu, sigma):
        if isinstance(sigma, (np.ndarray, list)):
            try:
                return np.random.choice(self._skewposterior(mu,sigma[0],sigma[1],self.N),self.N)
            except:
                return np.random.normal(mu, sigma[0], self.N)    
        else:
            return np.random.normal(mu, sigma, self.N)
    
    def _skewposterior(self, mu, sigma_up, sigma_lw, N):
        UP = np.random.normal(0,abs(sigma_up),size=N)
        LW = np.random.normal(0,abs(sigma_lw),size=N)
        return mu + np.concatenate([UP[UP>0],LW[LW<0]])

In [58]:
star = star(Fe=[0,0.1])

[-2, 0]
[-2, 0]
[-2, 0]
[-2, 0]
[-2, 0]
[-2, 0]


In [56]:
star.Fe

array([-2., -2., -2., ..., -2., -2., -2.], shape=(50000,))

In [ ]:
import numpy as np

class StarProperty:
    """
    Initialize base properties of a host star.
    Arrays are initialized for Monte Carlo sampling.
    """
    def __init__(self, N=50000, Mass=None, Radius=None, Teff=None, FeH=None, Age=None):
        self._N = N
        self._Mass = np.array(Mass) if Mass is not None else None
        self._Radius = np.array(Radius) if Radius is not None else None
        self._Teff = np.array(Teff) if Teff is not None else None
        self._FeH = np.array(FeH) if FeH is not None else None
        self._Age = np.array(Age) if Age is not None else None

    @property
    def N(self):
        return self._N

    @property
    def Mass(self):
        return self._Mass
    @Mass.setter
    def Mass(self, value):
        self._Mass = value

    @property
    def Radius(self):
        return self._Radius
    @Radius.setter
    def Radius(self, value):
        self._Radius = value

    @property
    def Teff(self):
        return self._Teff
    @Teff.setter
    def Teff(self, value):
        self._Teff = value

    @property
    def FeH(self):
        return self._FeH
    @FeH.setter
    def FeH(self, value):
        self._FeH = value

    @property
    def Age(self):
        return self._Age
    @Age.setter
    def Age(self, value):
        self._Age = value


class HostStar(StarProperty):
    """
    Wrapper to generate and compute stellar parameters using Monte Carlo 
    sampling for exoplanet host stars.

    Parameters:
    -----------
    N: int
        Number of samples to generate.
    Mass: list
        Star's mass in solar masses. Format: [mu, sigma] or [mu, sigma_up, sigma_lw].
    Radius: list
        Star's radius in solar radii. Format: [mu, sigma] or [mu, sigma_up, sigma_lw].
    Teff: list
        Effective temperature in Kelvin. Format: [mu, sigma] or [mu, sigma_up, sigma_lw].
    FeH: list
        Metallicity [Fe/H] in dex. Format: [mu, sigma] or [mu, sigma_up, sigma_lw].
    Age: list
        Stellar age in Gyr. Format: [mu, sigma] or [mu, sigma_up, sigma_lw].
    """

    def __init__(self, N=50000, Mass=[1.0, 0.05], Radius=[1.0, 0.05], **kwargs):
        # Dynamically set N based on input posteriors if provided
        N = len(Mass) if (isinstance(Mass, (list, np.ndarray)) and len(Mass) > 3) else int(N)
        N = len(Radius) if (isinstance(Radius, (list, np.ndarray)) and len(Radius) > 3) else int(N)
        
        super().__init__(N=N, **kwargs)
        
        # Populate distributions if standard [mu, sigma] inputs are given
        if isinstance(Mass, (list, np.ndarray)) and len(Mass) <= 3:
            self.set_Mass(mu=Mass[0], sigma=Mass[1:])
        elif isinstance(Mass, (list, np.ndarray)):
             self.Mass = np.array(Mass)

        if isinstance(Radius, (list, np.ndarray)) and len(Radius) <= 3:
            self.set_Radius(mu=Radius[0], sigma=Radius[1:])
        elif isinstance(Radius, (list, np.ndarray)):
             self.Radius = np.array(Radius)

    def set_Mass(self, mu=1.0, sigma=0.05):
        self.Mass = self._set_parameter(mu, sigma)

    def set_Radius(self, mu=1.0, sigma=0.05):
        self.Radius = self._set_parameter(mu, sigma)

    def set_Teff(self, mu=5778, sigma=50):
        self.Teff = self._set_parameter(mu, sigma)

    def set_FeH(self, mu=0.0, sigma=0.05):
        self.FeH = self._set_parameter(mu, sigma)

    def set_Age(self, mu=4.5, sigma=0.5):
        self.Age = self._set_parameter(mu, sigma)

    def _set_parameter(self, mu, sigma):
        """Generates the array of parameters based on symmetric or asymmetric errors."""
        if isinstance(sigma, (list, np.ndarray)) and len(sigma) == 2:
            return np.random.choice(self._skewposterior(mu, sigma[0], sigma[1], self.N), self.N)
        elif isinstance(sigma, (list, np.ndarray)) and len(sigma) == 1:
            return np.random.normal(mu, sigma[0], self.N)
        else:
            return np.random.normal(mu, float(sigma), self.N)
    
    def _skewposterior(self, mu, sigma_up, sigma_lw, N):
        """Constructs a split-normal (skewed) distribution."""
        UP = np.random.normal(0, abs(sigma_up), size=N)
        LW = np.random.normal(0, abs(sigma_lw), size=N)
        return mu + np.concatenate([UP[UP > 0], LW[LW < 0]])

    def _test(self):
        """Sanity checks to ensure physical boundaries are respected."""
        if self.Mass is None or self.Radius is None:
            raise ValueError('Mass and Radius must be set.')
        if len(self.Mass) != len(self.Radius):
            raise ValueError('Mass and Radius arrays must be of identical length.')
        if np.any(self.Mass <= 0) or np.any(self.Radius <= 0):
            raise ValueError('Mass and Radius must be strictly positive.')
        if self.Teff is not None and np.any(self.Teff <= 0):
            raise ValueError('Effective Temperature (Teff) must be strictly positive.')

    def corner(self, Data=['Mass', 'Radius', 'Teff', 'FeH'], bins=50, smooth=True, show_titles=True, **kwargs):
        """
        Corner plot of the stellar parameters.
        """
        try:
            import corner
        except ImportError:
            raise ImportError("corner.py is not installed. Please run: pip install corner")
            
        data_dict = self.__dict__
        corner_data = []
        labels = []
        
        for item in Data:
            # Check for property format (_Name)
            val = data_dict.get(f'_{item}', data_dict.get(item))
            if val is not None and np.any(val):
                corner_data.append(val)
                labels.append(item)
                
        if not corner_data:
            raise ValueError("No valid data found to plot. Ensure parameters are set.")
            
        corner_data = np.array(corner_data).T
        fig = corner.corner(corner_data, labels=labels, bins=bins, smooth=smooth, show_titles=show_titles, **kwargs)
        
        n = len(labels)
        axs = np.array(fig.axes).reshape((n, n))
        return fig, axs

In [44]:
star = host_star()

TypeError: host_star.__init__() missing 1 required positional argument: 'star_abundance'

In [ ]:


class rocky(exoplanet):
    '''
    Run the rocky planet model.

    Parameters:
    -----------
    star: list [Fe/H, Mg/H, Si/H]
        Stellar abundances of Fe/H, Mg/H, Si/H.
    ratio: str
        ratio to constrain the planet chemistry to the star.
        e.g. ratio='Fe/Si,Fe/Mg,Mg/Si' (default=None)
    star_norm: list [Fe/H, Mg/H, Si/H]
        Normalization reference for the stellar abundances. 
    
    Attributes:
    --------
    self.CMF: array
        Core mass fraction 
    self.FeMF: array
        Iron mass fraction
    self.SiMF: array
        Silicon mass fraction
    self.MgMF: array
        Magnesium mass fraction
    '''
    def __init__(self, Mass=[1,0.001], Radius=[1,0.001],  N=50000, **kwargs):
        '''
        Assuming purely rocky planet. xSi and xFe are free parameters.
        '''
        super().__init__(N, Mass, Radius,**kwargs)
        xSi = kwargs.get('xSi', [0,0.2])
        xFe = kwargs.get('xFe', [0,0.2])
        self.set_xSi(a=xSi[0], b=xSi[1])
        self.set_xFe(a=xFe[0], b=xFe[1])
        self._save_parameters = ['Mass','Radius','CMF','xSi','xFe','FeMF','SiMF','MgMF']
        self.type = 'rocky'
        self.Points, self.Radius_Data = get_cached_data(self.type) # load interpolation fits

    def run(self,star=None,ratio=None,star_norm=None):
        '''
        Running Monte Carlo simulation of rocky planet.
        '''
        get_R = lambda x: interpn(self.Points, self.Radius_Data, x) # x=cmf,Mass,xSi,xFe
        self._check(get_R)
        args = np.asarray([self.Radius,self.Mass,self.xSi,self.xFe]).T
        if star is None:
            residual = lambda x,param: np.sum(param[0]-get_R(np.asarray([x[0],*param[1:]]).T))**2/1e-6
            self.CMF = self._run_MC(residual,args)
            self.FeMF,self.SiMF,self.MgMF,_,_,_ = chemistry(self.CMF,xSi=self.xSi,xFe=self.xFe)
        elif ratio is None:
            warnings.warn('No target ratio provided. Running without stellar constraint.')
            residual = lambda x,param: np.sum(param[0]-get_R(np.asarray([x[0],*param[1:]]).T))**2/1e-6
            self.CMF = self._run_MC(residual,args)
            self.FeMF,self.SiMF,self.MgMF,_,_,_ = chemistry(self.CMF,xSi=self.xSi,xFe=self.xFe)
        else:
            if star_norm is None:
                star_norm = [7.46,7.55,7.51] # Fe, Mg, Si Asplund 2021
            mu = [55.85e-3,28.09e-3,24.31e-3] # Fe, Mg, Si atomic masses
            star_w = [10**(star[i]+star_norm[i]-12)*mu[i] for i in range(3)]
            
            ratio_split = ratio.lower().split(',') # Fe/Si, Fe/Mg, Mg/Si ....
            dr_star = {'fe': star_w[0],'mg': star_w[1], 'si': star_w[2]}
            print('Using stellar constraints:',end=' ')
            [print(item+f' ({eval(item, dr_star.copy()):.2f})',end=', ') for item in ratio_split]
            print()
            
            def residual(x, param):
                radius_residual = np.sum(param[0] - get_R(np.asarray([x[0],param[1],x[1],x[2]]).T))**2/1e-6
                data = chemistry(x[0], xSi=x[1], xFe=x[2],xWu=x[3])
                dr_planet = {'fe': data[0], 'si': data[1], 'mg': data[2]}
                chem_residual = 0
                for item in ratio_split:
                    chem_residual += np.sum(eval(item, dr_star.copy())-eval(item, dr_planet.copy()))**2/1e-6
                return radius_residual + chem_residual
            
            args = np.asarray([self.Radius,self.Mass]).T
            self.CMF,self.xSi,self.xFe,self.xWu = self._run_MC(residual,args,
                                xi=[0.325,0.1,0.1,0.2],bounds=[[0,1],[0,0.2],[0,0.2],[0,0.5]]).T
            self.FeMF,self.SiMF,self.MgMF,_,_,_ = chemistry(self.CMF,xSi=self.xSi,xFe=self.xFe,xWu=self.xWu)
    
class water(exoplanet):
    def __init__(self, Mass=[1,0.001], Radius=[1,0.001], N=50000, **kwargs):
        super().__init__(N, Mass, Radius, **kwargs)
        CMF = kwargs.get('CMF', [0.325,0.325])
        self.set_CMF(a=CMF[0], b=CMF[1])
        self._save_parameters = ['Mass','Radius','WMF','CMF']
        self.type = 'water'
        self.Points, self.Radius_Data = get_cached_data(self.type) # load interpolation fits

    def run(self):
        '''
        Running Monte Carlo simulation of water world.

        Note:
        ----
            The CMF parameter is assumed to be Rocky core mass fraction,
            such that rcmf = (1-wmf)/cmf
        '''
        get_R = lambda x: interpn(self.Points, self.Radius_Data, x) # x=wmf,Mass,cmf   
        self._check(get_R)
        args = np.asarray([self.Radius,self.Mass,self.CMF]).T
        residual = lambda x,param: np.sum(param[0]-get_R(np.asarray([x[0],param[1],param[2]]).T))**2/1e-6
        self.WMF = self._run_MC(residual,args)
        self.FeMF,self.SiMF,self.MgMF,_,_,_ = chemistry(self.CMF,xSi=0.,xFe=0.,xWu=0.2)

class envelope(exoplanet):
    def __init__(self, Mass=[1,0.001], Radius=[1,0.001], N=50000, **kwargs):
        super().__init__(N, Mass, Radius, **kwargs)
        # CMF not implemented yet
        # CMF = kwargs.get('CMF', [0.325,0.325]) 
        # self.set_CMF(a=CMF[0], b=CMF[1])
        Teq = kwargs.get('Teq', [1000,100])
        self.set_Teq(mu=Teq[0], sigma=Teq[1])
        self._save_parameters = ['Mass','Radius','AMF','Teq']
        self.type = 'envelope'
        self.Points, self.Radius_Data = get_cached_data(self.type) # load interpolation fits

    def run(self):
        '''
        Running Monte Carlo simulation for planet with H/He envelope.
    
        Note:
        ----
            The CMF parameter is fixed for the moment
        '''
        get_R = lambda x: interpn(self.Points, self.Radius_Data, x)
        self._check(get_R)
        args = np.asarray([self.Radius,self.Mass,self.Teq]).T
        residual = lambda x,param: np.sum(param[0]-get_R(np.asarray([x[0],param[1],param[2]]).T))**2/1e-6
        self.AMF = self._run_MC(residual,args,bounds=[[0.005,0.2]])
        # self.FeMF,self.SiMF,self.MgMF,_,_,_ = chemistry(self.CMF,xSi=0.,xFe=0.,xWu=0.2)


In [9]:
interior_ratio_constraints=['Fe/Si','Fe/Mg','Mg/Si']

In [23]:
fe_mass,si_mass,mg_mass,ca_mass,al_mass,ni_mass = 0.1,0.1,0.2,0.3,0.4,0.1
dr_planet = {'fe': fe_mass,'mg': mg_mass, 'si': si_mass,'ni': ni_mass, 'ca': ca_mass, 'al': al_mass}
for item in interior_ratio_constraints:
    print(eval(item.lower(),dr_planet))

1.0
0.5
2.0


In [24]:
for item in interior_ratio_constraints:
    print(eval(item.lower(),dr_planet))

1.0
0.5
2.0


In [31]:
xx = {'x':[0.,3,4],'y':[3,4,5]}
for item in xx.values():
    print(item[0])

0.0
3


In [7]:
param = [0,1,2,3]
def ff(x,a,b,c):
    print(b)
ff(param[0],*param[1:])

2


In [3]:
import exopie

In [6]:
exopie.rocky?

Init signature: exopie.rocky(Mass=[1, 0.001], Radius=[1, 0.001], N=50000, **kwargs)
Docstring:     
`   Calculate the interior structure parameters (CMF, Fe-MF, WMF) for a given exoplanet 
    using the SUPEREARTH interior structure model developed by Valencia et al. (2006) 
    and updated in Plotnykov & Valencia (2020) as well as the AMF using the H2-He grid 
    based on results using CEPAM (Guillot & Morel 1995; Guillot 2010) and H-He EOS from Saumon et al. (1995).

    This function constructs a predictive model by interpolating the data to estimate 
    the interior based on the mass and radius for rocky, water or thin 
    envelope models. The methodology is detailed in Plotnykov & Valencia (2024).

    Parameters:
    -----------
    N: int
        Number of samples to generate. 
        If Mass or Radius posterior is given, N is set to the size [n].
    Mass: list
        Set planet's mass in Earth masses, 
        assumes normal distribution or skew normal distribution.
     

In [ ]:
import exopie
import numpy as np

# find the rocky threshold radius (RTR, cmf=0) at 5 Earth masses
R = exopie.get_radius(5,xSi=0,xFe=0,cmf=0.)
print(f'Radius Threshold (at 5Me) = {R[0]:.2f} Re')

# Find the interior of M=5 +/-0.5, R=1.4 +/-0.05 planet
planet = exopie.rocky(N=50000,Mass=[5,0.5],Radius=[1.4,0.05]) #model input
planet.run() # run the model

print(f'CMF = {np.mean(planet.CMF):.2f}', 
      f'FeMF = {np.mean(planet.FeMF):.2f}')

# Case where no Si in the core (xSi=0) and no Fe in the mantle (xFe=0)
planet = exopie.rocky(N=50000,Mass=[5,0.5],Radius=[1.4,0.05],
                        xSi=[0,0],xFe=[0,0]) # model input
planet.run() # run the model
print(f'CMF = {np.mean(planet.CMF):.2f}', 
      f'FeMF = {np.mean(planet.FeMF):.2f}')

planet.save_data('rocky.pkl') # save the model


# To make a nice corner plot use (need corner.py)
# fig,axs = planet.corner()

